# Code Book — Week 4, Anomaly Detection I

**Programme:** Applied Machine Learning for Operations — Model Builder Track
**Week:** 4 — Anomaly Detection I, Statistical Methods
**AI-use mode:** AI-Tutor. The assistant may explain and debug; you write the detector.
**Files:** 5 CSVs in `data/`

---

## What this week needs from the data

Week 3 asked what a series *is*. Week 4 asks when it is *wrong*. That needs three things the Week 3 files did not have to provide:

1. **A stable normal.** You cannot define an anomaly without a baseline. The bus power file is deliberately flat outside eclipse season for exactly this reason.
2. **Anomalies of more than one shape.** A spike, a level shift and a drift are three different problems. All three are present, in three different files.
3. **Ground truth.** Without a label file, "did the detector work?" is a matter of opinion. `telemetry_anomaly_labels.csv` settles it.

## Provenance

These are **synthetic teaching datasets**, not published NIGCOMSAT records. They were specified to reproduce the real behaviour of a GEO satellite at 42.5°E — eclipse seasons at the equinoxes, Ka-band rain attenuation over a tropical climate, gradual battery ageing — so the methods land on data that behaves like your telemetry.

Because they contain no real operational data, they may be used freely in AI assistants. Real NIGCOMSAT telemetry remains subject to the Week 2 data restrictions.

**Instructors:** the label file is the answer key. Withhold it until participants have run their own detectors.

## The five files

| File | Cadence | Rows | Anomaly shapes present | Used in |
|---|---|---|---|---|
| `bus_power_telemetry_hourly.csv` | hourly | 2,880 | 3 spikes | 4.1, 4.2, 4.3, 4.5 |
| `twta_thermal_daily.csv` | daily | 730 | 1 spike, 1 level shift | 4.2, 4.3, 4.4, 4.5 |
| `battery_health_daily.csv` | daily | 1,096 | 1 drift (fade knee) | 4.1, 4.4, 4.5 |
| `rf_link_quality_hourly.csv` | hourly | 744 | none injected — real rain fade | 4.1, 4.5 |
| `telemetry_anomaly_labels.csv` | — | 6 | ground truth for all of the above | 4.5 |

### Why each file is here

**`bus_power_telemetry_hourly`** — the primary detection target. Outside eclipse season the platform sits in continuous sunlight and the telemetry is genuinely stable, so control limits are unambiguous and a deviation means something. It also carries the clearest false-alarm trap in the collection: a fixed floor on state of charge fires on all 46 eclipse hours, every one of them correct and useless.

**`twta_thermal_daily`** — drift and level-shift work. Its trend is confounded by a rising RF drive level, so the naive reading ("the amplifier is overheating") mistakes increased load for ageing. The level shift is 6.9σ against local spread but invisible to a single-point 3σ rule, which is precisely the case CUSUM exists for.

**`battery_health_daily`** — the case where a spike detector finds nothing at all. Capacity falls for three years without one day breaching a fixed limit, and the *rate* doubles in April 2025. The event is a change in slope, not a value.

**`rf_link_quality_hourly`** — the false-alarm lesson. Twelve hours genuinely breach, buried in 56 hours of rain-fade noise. A naive threshold on C/N cannot separate "raining" from "link down".

**`telemetry_anomaly_labels.csv`** — ground truth, so 4.5 can compute precision and recall rather than argue about them.

## Setup

Run this first. It assumes the notebook sits next to the `data/` folder.

**What this cell does.** Imports pandas, numpy and matplotlib, points `DATA` at the `data/` folder beside this notebook, and lists the five CSVs. Run it first. If the list comes back empty or short, the notebook is not sitting next to `data/`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA = Path("data")
pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 25)

sorted(p.name for p in DATA.glob("*.csv"))

---

## 1. Bus power and battery telemetry — hourly

`data/bus_power_telemetry_hourly.csv`

Electrical power subsystem telemetry across a full eclipse season. A GEO satellite runs off its solar arrays in sunlight; twice a year the Earth's shadow crosses the orbit and it runs on battery for up to 72 minutes a day. This file captures that at hourly resolution, which gives it a clean 24-hour period inside a seasonal envelope.

**Shape:** 2,880 rows — 2025-08-01T00:00 to 2025-11-28T23:00

### Columns

| Column | Type | Unit | Description |
|---|---|---|---|
| `timestamp` | datetime (hourly) | — | Hour of the reading, UTC |
| `solar_array_current_a` | float | A | Array output. ~27.8 A in sunlight, ~0 in eclipse, elevated during recharge |
| `bus_voltage_v` | float | V | Regulated bus voltage. ~50.4 V nominal, sagging to ~46 V on battery |
| `battery_soc_pct` | float | % | State of charge. Falls to about 77% at the deepest eclipse |
| `battery_temp_c` | float | °C | Pack temperature. ~18 °C nominal, rising on discharge and recharge |
| `eclipse_flag` | int | 0/1 | 1 when the satellite is in eclipse that hour |

### Notes and known behaviour

- **Eclipse season runs 2025-09-01 to 2025-10-16** — 46 eclipse hours. Outside those dates the series is nearly flat, which is what makes it a good control-limit target.
- Eclipse occurs at **21:00 UTC**, near satellite local midnight at 42.5°E. Period is 24, not 7.
- Discharge depth tracks eclipse duration, so the season builds and fades rather than switching on.
- `eclipse_flag` is the honest way to suppress seasonal false alarms — condition on it rather than widening the limit.
- **Three injected spikes** (EV01–EV03). Array current drops to ~55% of expected with battery temperature ~4.6 °C high.

**What to read in the output.** 2,880 hourly rows, no gaps, 46 eclipse hours. The two lines below are the ones that matter for 4.2: the sunlit baseline has a mean near 27.8 A and a standard deviation around 0.06 A, which puts the 3σ band roughly 27.6–28.0 A. That band is narrow because the platform genuinely is stable in sunlight — and that is why a deviation means something.

In [ ]:
bus = pd.read_csv(DATA / "bus_power_telemetry_hourly.csv",
                  parse_dates=["timestamp"], index_col="timestamp").asfreq("H")

print("rows:", len(bus), " gaps:", int(bus.isna().sum().sum()))
print("eclipse hours:", int(bus["eclipse_flag"].sum()))

# the baseline every detector in 4.2 is built on
quiet = bus[bus["eclipse_flag"] == 0]["solar_array_current_a"]
print(f"\nsunlit baseline: mean={quiet.mean():.2f} A  sd={quiet.std():.3f} A")
print(f"3-sigma band   : {quiet.mean()-3*quiet.std():.2f} to {quiet.mean()+3*quiet.std():.2f} A")

---

## 2. TWTA thermal — daily

`data/twta_thermal_daily.csv`

Two years of travelling-wave tube amplifier temperature with the RF drive level that partly explains it. Ageing drift, a yearly sun-angle cycle and a rising drive level all act on one column — plus an injected level shift. It is the hardest file here to interpret, and the most representative of real subsystem data.

**Shape:** 730 rows — 2024-01-01 to 2025-12-30

### Columns

| Column | Type | Unit | Description |
|---|---|---|---|
| `date` | date | — | Calendar day |
| `twta_temp_c` | float | °C | Amplifier baseplate temperature, roughly 59–69 |
| `rf_drive_pct` | float | % | RF drive as a share of saturation, rising from ~68% to ~82% |

### Notes and known behaviour

- **Four effects in one column:** ageing drift of ~+0.003 °C/day, a yearly sun-angle cycle of ±1.6 °C, a contribution from rising drive level, and the injected shift.
- **Level shift EV05:** +1.8 °C from 2025-05-12, sustained to the end of the file. Measured against local spread it is 6.9σ, but no single reading looks extreme in the context of the whole series — this is the CUSUM case.
- **Spike EV04:** 2024-09-03 to 09-04, a transient excursion of about +4.9 °C then +3.2 °C.
- `twta_temp_c` and `rf_drive_pct` correlate strongly, and most of it is shared trend. Difference both before comparing.

**What to read in the output.** First the level shift: the thirty days before 2025-05-12 average about 65.2 °C, the thirty after about 67.4 °C — a 2.2 °C jump, which against a local spread of 0.32 °C is roughly 6.9σ. Then the trap: raw correlation between temperature and drive level is high, and after differencing it largely disappears. Most of that correlation was shared trend, not physics.

In [ ]:
twta = pd.read_csv(DATA / "twta_thermal_daily.csv",
                   parse_dates=["date"], index_col="date").asfreq("D")

print("rows:", len(twta))

# the level shift, measured against local spread
pre  = twta.loc["2025-04-12":"2025-05-11", "twta_temp_c"]
post = twta.loc["2025-05-12":"2025-06-10", "twta_temp_c"]
print(f"\npre-shift  : {pre.mean():.2f} C  (sd {pre.std():.2f})")
print(f"post-shift : {post.mean():.2f} C")
print(f"jump       : {post.mean()-pre.mean():.2f} C = {(post.mean()-pre.mean())/pre.std():.1f} sigma")

# the trap
print(f"\nraw correlation with drive : {twta['twta_temp_c'].corr(twta['rf_drive_pct']):.3f}")
print(f"differenced correlation    : {twta['twta_temp_c'].diff().corr(twta['rf_drive_pct'].diff()):.3f}")

---

## 3. Battery health — daily

`data/battery_health_daily.csv`

Three years of daily battery condition. Capacity fades steadily, then the fade rate roughly doubles in April 2025. No single day is unusual — the anomaly is entirely in the slope, which makes this the file that defeats every point-wise detector.

**Shape:** 1,096 rows — 2023-01-01 to 2025-12-31

### Columns

| Column | Type | Unit | Description |
|---|---|---|---|
| `date` | date | — | Calendar day |
| `eclipse_minutes` | float | min | Eclipse duration that day. 0 outside seasons, up to 72 at equinox |
| `depth_of_discharge_pct` | float | % | How far the battery was drawn down |
| `eod_voltage_v` | float | V | End-of-discharge voltage. Falls with both discharge depth and age |
| `est_capacity_ah` | float | Ah | Estimated usable capacity. About 103 Ah falling to 95.4 Ah |
| `cycle_count` | int | count | Cumulative cycles, starting from 4,180 |

### Notes and known behaviour

- **Drift EV06:** fade rate roughly doubles from 2025-04-01. Pre-knee loss is about −2.4 Ah/year; post-knee about −4.0 Ah/year.
- **Two seasonal peaks a year**, around 21 March and 23 September — do not assume one annual hump.
- No day breaches a 95 Ah limit within the file. A threshold detector scores zero here, which is the point.
- Between eclipse seasons `depth_of_discharge_pct` sits near 0.4–0.9% rather than exactly zero. That is housekeeping load, not missing data.

**What to read in the output.** Capacity falls from about 103 Ah to 95.4 Ah and never breaches 95 Ah — so a threshold detector fires zero times across three years. The two fade rates at the end are the actual event: roughly −2.4 Ah/year before April 2025 and about −4.0 Ah/year after. Nothing about any individual day is unusual; the slope changed.

In [ ]:
bat = pd.read_csv(DATA / "battery_health_daily.csv",
                  parse_dates=["date"], index_col="date").asfreq("D")

cap = bat["est_capacity_ah"]
print(f"capacity: start {cap.iloc[0]:.1f} Ah  ->  end {cap.iloc[-1]:.1f} Ah")
print(f"never below 95 Ah? {(cap < 95).sum() == 0}")

# the anomaly is in the slope, not the level
def fade_rate(s):
    x = np.arange(len(s))
    return np.polyfit(x, s.values, 1)[0] * 365

print(f"\nfade rate before Apr 2025 : {fade_rate(cap.loc['2024-01-01':'2025-03-31']):.2f} Ah/year")
print(f"fade rate after  Apr 2025 : {fade_rate(cap.loc['2025-04-01':]):.2f} Ah/year")

---

## 4. RF link quality — hourly, August 2025

`data/rf_link_quality_hourly.csv`

One month of Ka-band link telemetry at the wettest point of the rainy season. No anomalies are injected: every excursion is genuine rain fade. That is what makes it the false-alarm exercise — the detector must separate "heavy rain, link holding" from "link actually down".

**Shape:** 744 rows — 2025-08-01T00:00 to 2025-08-31T23:00

### Columns

| Column | Type | Unit | Description |
|---|---|---|---|
| `timestamp` | datetime (hourly) | — | Hour of the reading, UTC |
| `rain_rate_mmhr` | float | mm/hr | Rain rate at the earth station, 0 to about 58 |
| `cn_ratio_db` | float | dB | Carrier-to-noise ratio. About 14.25 dB in clear sky |
| `uplink_power_dbm` | float | dBm | Transmitted uplink power, with about 6 dB of headroom |
| `link_margin_db` | float | dB | `cn_ratio_db` minus the 8.5 dB threshold. Negative means the link is down |

### Notes and known behaviour

- **56 hours carry rain. 16 fall below 2 dB of margin. 12 go negative** — those twelve are the only true outages.
- Rain attenuation follows a power law, so heavy storms remove far more signal than 6 dB of uplink power control can restore.
- `uplink_power_dbm` is a *response* to rain, not an independent measurement. Feeding it to a detector as a feature leaks the answer.
- A naive 3σ rule on `cn_ratio_db` flags roughly all 56 rainy hours — a 79% false-alarm rate against the 12 real outages. That number is the headline for 4.5.

**What to read in the output.** 56 hours carry rain, 16 drop below 2 dB, and 12 go negative — only those twelve are genuine outages. Then the naive detector: a 3σ rule on C/N flags far more hours than there are real outages, because heavy rain and a dropped link look similar in that column. The gap between "flagged" and "of which real" is the false-alarm problem 4.5 quantifies.

In [ ]:
rf = pd.read_csv(DATA / "rf_link_quality_hourly.csv",
                 parse_dates=["timestamp"], index_col="timestamp").asfreq("H")

print("hours with rain  :", int((rf["rain_rate_mmhr"] > 0).sum()))
print("hours below 2 dB :", int((rf["link_margin_db"] < 2).sum()))
print("hours below 0 dB :", int((rf["link_margin_db"] < 0).sum()), " <- the true outages")

# what a naive 3-sigma rule on C/N would flag
cn = rf["cn_ratio_db"]
flagged = cn < cn.mean() - 3 * cn.std()
true_out = rf["link_margin_db"] < 0
print(f"\n3-sigma flags     : {int(flagged.sum())}")
print(f"of which real     : {int((flagged & true_out).sum())}")

---

## 5. Anomaly ground truth — the answer key

`data/telemetry_anomaly_labels.csv`

Six labelled events across three telemetry files, with start, end, shape and operational cause. This is what turns "the detector seems to work" into a precision and recall figure. Withhold it from participants until they have run their own detectors.

**Shape:** 6 rows — one per event

### Columns

| Column | Type | Unit | Description |
|---|---|---|---|
| `event_id` | string | — | EV01 to EV06 |
| `source_file` | string | — | Which CSV the event lives in |
| `column` | string | — | Which column carries it |
| `start` | datetime | — | First affected timestamp |
| `end` | datetime | — | Last affected timestamp. Equals `start` for single-point spikes |
| `type` | string | — | `spike`, `level_shift` or `drift` |
| `severity` | string | — | `minor` or `major` |
| `operational_cause` | string | — | The plain-language explanation an engineer would write in the log |

### Notes and known behaviour

- **Three spikes** in bus power (EV01–EV03), **one spike and one level shift** in TWTA thermal (EV04–EV05), **one drift** in battery capacity (EV06).
- Two events span more than one timestamp — EV02 covers two hours, EV05 covers seven months. Scoring must decide whether a detector that flags one day of a seven-month shift counts as a hit. Make that rule explicit before computing recall.
- The RF file has no labelled events by design: everything in it is genuine rain fade, so it measures false alarms rather than detection.
- Roughly 6 events against 4,700 observations. That imbalance is why accuracy is a useless metric here and precision/recall are not.

**What to read in the output.** All six events in one table. Note the two that span time: EV02 covers two hours, EV05 covers seven months. Those force the scoring decision on the next slide — does flagging one day inside a seven-month shift count as catching it? Decide before you compute recall, not after.

In [ ]:
labels = pd.read_csv(DATA / "telemetry_anomaly_labels.csv",
                     parse_dates=["start", "end"])
labels

---

## Scoring conventions for the weekly deliverable

Agree these before you compute anything, and state them in your write-up.

| Decision | Recommended convention |
|---|---|
| Does flagging part of a multi-hour event count as a hit? | Yes — one detection anywhere inside `start`–`end` counts as catching that event |
| Does flagging the same event twice count twice? | No — collapse consecutive flags into one detection |
| How close is close enough? | Within ±1 sampling interval of the labelled window |
| Which metric leads? | Recall for safety-critical subsystems, precision where alarm fatigue is the risk |
| Why not accuracy? | 6 events in ~4,700 observations. A detector that flags nothing scores 99.9% |

### A reusable event-level scorer

**What this cell does.** Defines `score_events()`, the scorer used for the weekly deliverable. It works at event level rather than per row: for each labelled event it asks whether the detector flagged anything inside the window (plus a tolerance), counts that as one catch, then counts every flag outside any labelled window as a false alarm. Returns events, catches, false alarms, recall and precision. The example at the end scores the naive threshold from 4.1 — expect recall near zero and a large false-alarm count.

In [ ]:
def score_events(flags, labels, source_file, column, tol="1H"):
    """Event-level precision and recall.

    flags  : boolean Series indexed by timestamp
    labels : the label DataFrame
    """
    ev = labels[(labels["source_file"] == source_file) &
                (labels["column"] == column)]
    tol = pd.Timedelta(tol)

    caught = 0
    matched = pd.Series(False, index=flags.index)
    for _, row in ev.iterrows():
        window = (flags.index >= row["start"] - tol) & (flags.index <= row["end"] + tol)
        if flags[window].any():
            caught += 1
        matched |= window

    false_alarms = int((flags & ~matched).sum())
    recall = caught / len(ev) if len(ev) else float("nan")
    detections = caught + false_alarms
    precision = caught / detections if detections else float("nan")

    return {"events": len(ev), "caught": caught,
            "false_alarms": false_alarms,
            "recall": round(recall, 3), "precision": round(precision, 3)}


# example: the naive threshold from 4.1, scored properly
bus_flags = bus["battery_soc_pct"] < 82.0
score_events(bus_flags, labels, "bus_power_telemetry_hourly.csv", "solar_array_current_a")

---

## Change log

| Date | Change |
|---|---|
| 2026-09-08 | First issue for Week 4. Added `telemetry_anomaly_labels.csv`; injected a level shift and transient spike into TWTA thermal, and a fade knee into battery capacity. |

*Applied Machine Learning for Operations, Model Builder Track — prepared for NIGCOMSAT by Publica Academy.*